# v0.2.9 live integration: general pipeline on the frozen v0.2.8 engine

**Code under test:** branch `v0.2.9-pipeline`. **Immutable baseline:** `v0.2.8-contract-frozen` (commit `3d01100`). **Earth Engine project:** `clever-grid-229708`.
You only interact with this notebook. Nothing needs to be copied or run outside it.

| Order | Cells | What it does | When |
|---|---|---|---|
| Setup | 1-5 | clones the branch, **verifies the frozen engine closure (19 files) byte-for-byte and its SHA-256**, sets the Earth Engine project, verifies the configuration, loads the four zones | always |
| **REFERENCE REPRODUCIBILITY DIAGNOSTIC** | D1, D2, D3 | tests whether the reference mismatch found by the first live run is explained by the date the reference was built; **no input needed** | now (about 15-20 min) |
| Part A | A1-A4 | the four live Tata zones through the pipeline, with engine-level parity against the standalone audits | **later**, after the diagnostic is resolved |
| Part B | B1-B4 | SoulForest, seven zones, all terrestrial | **later**, after Part A passes |

**Start on a fresh runtime** (Runtime -> Disconnect and delete runtime, then reconnect) and run cells in order; Earth Engine will ask you to authenticate once in cell 3.
Any failed gate raises and stops the notebook. Nothing printed here is a validation result by itself.

In [ ]:
# 1. Clone the pipeline branch, then verify the FROZEN ENGINE CLOSURE (the 19 modules that determine every result) is byte-identical to the frozen tag
#    BEFORE importing anything or touching Earth Engine.
import os, glob, subprocess, hashlib
REPO_URL = 'https://github.com/G-auravSingh/reference-benchmarking.git'
BRANCH = 'v0.2.9-pipeline'
FROZEN_TAG = 'v0.2.8-contract-frozen'
FROZEN_COMMIT_PREFIX = '3d01100'
FROZEN_ENGINE_CLOSURE_SHA256 = '56b57ebe4d62fb804c966c504c3f8776e62954f378fdf8f27b8757f4ff4c6ed8'   # computed from git at the frozen tag over the 19-file executable engine closure
ENGINE_CLOSURE_FILES = ('__init__.py',
                         'assess.py',
                         'benchmarking.py',
                         'config.py',
                         'constructs.py',
                         'contracts.py',
                         'ecoregion.py',
                         'estimators.py',
                         'indicator_contract.py',
                         'indicators/__init__.py',
                         'pipeline.py',
                         'reference.py',
                         'reference_builders_ee.py',
                         'registry.py',
                         'scoring.py',
                         'site_loader.py',
                         'statistics.py',
                         'support.py',
                         'tiling.py')
REPO_DIR = '/content/reference-benchmarking'

def git(*a, text=True):
    r = subprocess.run(['git', '-C', REPO_DIR, *a], capture_output=True, text=text)
    if r.returncode != 0:
        raise RuntimeError(f'git {a} failed: {r.stderr}')
    return r.stdout.strip() if text else r.stdout

if os.path.exists(REPO_DIR):          # plain 'restart session' keeps the disk: accept ONLY a clean clone that equals the remote branch head
    git('fetch', '-q', 'origin', BRANCH)
    assert git('rev-parse', 'HEAD') == git('rev-parse', f'origin/{BRANCH}'), 'existing clone is not at the remote branch head: delete the runtime and start fresh'
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', 'origin', 'tag', FROZEN_TAG, '--no-tags'], check=True)

branch = git('rev-parse', '--abbrev-ref', 'HEAD'); commit = git('rev-parse', 'HEAD')
frozen_commit = git('rev-parse', f'{FROZEN_TAG}^{{commit}}')
status = git('status', '--porcelain', '--untracked-files=no')
PKG_DIR = sorted(glob.glob(f'{REPO_DIR}/darukaa_reference_v*'))[-1]
PKG_REL = os.path.relpath(PKG_DIR, REPO_DIR)

assert branch == BRANCH, f'WRONG BRANCH: {branch}'
assert not status, f'working tree is not clean: {status}'
assert frozen_commit.startswith(FROZEN_COMMIT_PREFIX), f'frozen tag points to {frozen_commit}, expected {FROZEN_COMMIT_PREFIX}'
subprocess.run(['git', '-C', REPO_DIR, 'merge-base', '--is-ancestor', frozen_commit, commit], check=True)   # the pipeline branch is built on the frozen commit

h = hashlib.sha256(); changed = []
for f in sorted(ENGINE_CLOSURE_FILES):
    frozen_bytes = git('show', f'{FROZEN_TAG}:{PKG_REL}/darukaa_reference/{f}', text=False)
    here_bytes = open(f'{PKG_DIR}/darukaa_reference/{f}', 'rb').read()
    if frozen_bytes != here_bytes:
        changed.append(f)
    h.update(f.encode()); h.update(here_bytes)
engine_closure_sha256 = h.hexdigest()
assert not changed, f'FROZEN ENGINE FILES DIFFER FROM {FROZEN_TAG}: {changed}  -> STOP'
assert engine_closure_sha256 == FROZEN_ENGINE_CLOSURE_SHA256, f'engine SHA-256 {engine_closure_sha256} != frozen {FROZEN_ENGINE_CLOSURE_SHA256}  -> STOP'

print('repository          :', REPO_URL)
print('branch              :', branch)
print('pipeline commit     :', commit)
print('frozen engine commit:', frozen_commit, f'({FROZEN_TAG})')
print('engine closure SHA-256:', engine_closure_sha256, '== frozen: True  (all', len(ENGINE_CLOSURE_FILES), 'engine-closure files byte-identical to the tag)')
print('working tree        : clean')
print('package             :', PKG_DIR)

In [ ]:
# 2. Install (same pattern as the frozen-tag notebook)
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pyproj shapely

In [ ]:
# 3. Earth Engine: explicit project, no fallback.
import ee
GEE_PROJECT = 'clever-grid-229708'
FORBIDDEN_GEE_PROJECT = 'gaurav-singh-007'
assert GEE_PROJECT != FORBIDDEN_GEE_PROJECT
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
assert ee.Number(1).getInfo() == 1
try:
    _active = ee.data._get_state().cloud_api_user_project
except Exception:
    _active = None
if _active is not None:
    assert _active == GEE_PROJECT, f'Earth Engine is using {_active!r}, expected {GEE_PROJECT!r}'
print('Active Earth Engine project:')
print(GEE_PROJECT, '(verified by the client state)' if _active else '(initialised with this project; client state not introspectable)')

In [ ]:
# 4. Configuration and PROVENANCE GATE. Identities are recorded SEPARATELY (commits, SHAs, versions, project).
import sys, json, subprocess
sys.path.insert(0, PKG_DIR); os.chdir(PKG_DIR)
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
from darukaa_reference import assess as A, provenance as PV, engine_identity as EI, engine_pipeline as EP, indicator_contract as IC, manifest as M

config = Config(                      # the frozen standalone configuration, with ONLY the Earth Engine project different (gee_project is not read by the engine)
    gee_project=GEE_PROJECT, output_dir='outputs', archetype='industrial', realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True, reference_stratification='ecoregion_landcover')
registry = create_default_registry()

assert EI.engine_closure_sha256() == EI.FROZEN_ENGINE_CLOSURE_SHA256 == FROZEN_ENGINE_CLOSURE_SHA256, 'imported engine differs from the frozen engine  -> STOP'
prov = EP.check_run_provenance(config, strict=True)        # raises on a non-frozen engine, a dirty/moved checkout, or an engine-config difference
assert prov['engine']['engine_closure_matches_frozen'] and prov['engine']['engine_config_matches_frozen'] and not prov['warnings'], prov['warnings']
assert not subprocess.run(['git', '-C', REPO_DIR, 'status', '--porcelain', '--untracked-files=no'], capture_output=True, text=True).stdout.strip(), 'tree became dirty'

IDENTITY = {
    'frozen_engine_commit': frozen_commit, 'frozen_tag': FROZEN_TAG,
    'pipeline_commit': commit, 'pipeline_branch': branch, 'pipeline_version': EP.PIPELINE_VERSION,
    'engine_closure_sha256': engine_closure_sha256, 'engine_closure_sha256_matches_frozen': True, 'engine_closure_files': list(EI.ENGINE_CLOSURE_FILES), 'executed_non_result_files': list(EI.EXECUTED_NON_RESULT_FILES),
    'pipeline_source_sha256': prov['source_sha256_at_import'],
    'contract_version': IC.CONTRACT_VERSION, 'engine_code_version': A.CODE_VERSION,
    'config_sha256': prov['config_sha256'], 'engine_config_sha256': prov['engine']['engine_config_sha256'],
    'frozen_engine_config_sha256': EI.FROZEN_ENGINE_CONFIG_SHA256, 'engine_config_matches_frozen': True,
    'gee_project': GEE_PROJECT, 'working_tree': 'clean'}
for k, v in IDENTITY.items():
    print(f'{k:30s}: {v}')
print()
print('NOTE: config_sha256 differs from the frozen standalone value (7f64f9f0...) ONLY because gee_project differs; the engine-config SHA-256, which covers the fields the engine reads, is identical.')

In [ ]:
# 5. Load the four Tata zones (realms taken from the manifest's explicit tile_realms, not from names) and the standalone v0.2.8 baseline. NOTHING TO ENTER.
import logging, time
EXPECTED_TATA = {'Pond_4': 'aquatic', 'Pond_2': 'aquatic', 'Pond_3': 'aquatic', 'EMU_Wetland_forest': 'terrestrial'}   # the intended assignments, checked against the manifest
REPO_ROOT = REPO_DIR
_warn = []
class _Cap(logging.Handler):
    def emit(self, record): _warn.append(record.getMessage())
_h = _Cap(level=logging.WARNING); logging.getLogger('darukaa_reference.manifest').addHandler(_h)
tata_project, tata_specs_all = M.resolve_zone_specs(M.find_manifest_by_project_name(REPO_ROOT, 'TataMotors_Pimpri'), REPO_ROOT)
logging.getLogger('darukaa_reference.manifest').removeHandler(_h)
assert not _warn, f'legacy realm/companion INFERENCE was used: {_warn}'
by_label = {s.label: s for s in tata_specs_all}
tata_specs = [by_label[l] for l in EXPECTED_TATA]
for s in tata_specs:
    assert s.realm == EXPECTED_TATA[s.label], f'{s.label}: manifest says {s.realm}, expected {EXPECTED_TATA[s.label]}'
    assert s.realm_source == 'manifest', f'{s.label}: realm was not explicitly declared ({s.realm_source})'
    print(f'{s.label:22s} realm={s.realm:12s} declared by: {s.realm_source:9s} from manifest project: {s.project}   tile: {os.path.relpath(s.path, REPO_DIR)}')
STANDALONE_DIR = f'{PKG_DIR}/tests/fixtures/tata_v0_2_8_standalone'      # the completed standalone v0.2.8 audits (frozen-tag run)
assert all(os.path.exists(f'{STANDALONE_DIR}/{l}_audit.json') for l in EXPECTED_TATA)
print('\nstandalone baseline:', STANDALONE_DIR)


# Drive (outputs are written here automatically) and run folder names
import pandas as pd, time
from datetime import date, datetime, timedelta
from google.colab import drive
drive.mount('/content/drive')
RUN_TAG = 'run1'                          # change only to repeat a run into a fresh folder
OUT_ROOT = '/content/drive/MyDrive/Darukaa/v0.2.9_live_validation'
os.makedirs(OUT_ROOT, exist_ok=True)

# 8a. Parity machinery (definitions only). ENGINE-LEVEL comparison of every field of every row, live vs the standalone v0.2.8 audit.
#     Tolerance: exact for strings/ints/None/bools; floats must agree to a relative 1e-12 (the tolerance established for the offline parity tests;
#     the engine was found to reproduce earlier live runs exactly). Anything else is UNEXPECTED, printed in full, and stops the notebook.
#     EXPECTED differences (layer / run identity, not engine results) are exactly:  row 'seconds'  and  row provenance.git_commit  (3d01100 vs the pipeline commit).
import json, math
REL_TOL = 1e-12
def _isnum(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def deep_diff(a, b, path, out=None):
    out = [] if out is None else out
    if isinstance(a, str) and isinstance(b, str) and a != b:
        try:
            ja, jb = json.loads(a), json.loads(b)
        except Exception:
            out.append((path, a, b, None)); return out
        if isinstance(ja, (dict, list)) and isinstance(jb, (dict, list)):
            return deep_diff(ja, jb, path, out)
        out.append((path, a, b, None)); return out
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b)):
            if k not in a or k not in b:
                out.append((f'{path}.{k}', a.get(k, '<absent>'), b.get(k, '<absent>'), None))
            else:
                deep_diff(a[k], b[k], f'{path}.{k}', out)
        return out
    if isinstance(a, list) and isinstance(b, list):
        if len(a) != len(b):
            out.append((path + '[len]', len(a), len(b), None)); return out
        for i, (x, y) in enumerate(zip(a, b)):
            deep_diff(x, y, f'{path}[{i}]', out)
        return out
    if a is None or b is None:
        if not (a is None and b is None): out.append((path, a, b, None))
        return out
    if isinstance(a, bool) or isinstance(b, bool):
        if a is not b: out.append((path, a, b, None))
        return out
    if _isnum(a) and _isnum(b):
        if a != b and not math.isclose(a, b, rel_tol=REL_TOL, abs_tol=0.0):
            out.append((path, a, b, abs(a - b) / max(abs(a), abs(b))))
        return out
    if a != b: out.append((path, a, b, None))
    return out

def compare_audits(live, sa):
    """-> (unexpected, expected, n_fields_compared). live / sa are the parsed *_audit.json files."""
    unexpected, expected, n = [], [], 0
    if live['code_version'] != sa['code_version']:
        unexpected.append(('code_version', live['code_version'], sa['code_version'], None))
    lr, sr = {r['indicator']: r for r in live['rows']}, {r['indicator']: r for r in sa['rows']}
    if set(lr) != set(sr):
        unexpected.append(('indicator_set', sorted(set(lr) ^ set(sr)), None, None))
    for ind in sorted(set(lr) & set(sr)):
        for k in sorted(set(lr[ind]) | set(sr[ind])):
            n += 1
            if k == 'seconds':
                expected.append((f'{ind}.seconds', lr[ind].get(k), sr[ind].get(k), None)); continue
            for d in deep_diff(lr[ind].get(k, '<absent>'), sr[ind].get(k, '<absent>'), f'{ind}.{k}'):
                (expected if d[0] == f'{ind}.provenance.git_commit' else unexpected).append(d)
    for d in deep_diff(live['meta'], sa['meta'], 'meta'):
        unexpected.append(d)
    return unexpected, expected, n

# Difference grouping (a summary, not a dump): zone x indicator x field category
import re as _re
_CATS = [
    ('geometry_derived (float noise)', r'reference_construction\.(site_area_m2|cell_area_ratio_to_site|cell_area_m2)$'),
    ('site_value', r'\.(site_value|site_unit|site_support)$|\.provenance\.site_meta'),
    ('reference_n', r'\.reference_n$|benchmark_details\.reference_n$'),
    ('reference_population', r'\.reference_population$|\.population$|\.population_taken$|reference_construction\.(eligible_cells_total|leaf_summary|allocation_min|allocation_max|n_returned|n_requested_total|shortfall|shortfall_note|n_tiles|n_leaf_tiles|n_tile_splits|landcover_class|hmi_threshold|population)'),
    ('reference_statistic', r'\.(reference_median|reference_mad|reference_p\d+|reference_unit|reference_support)$|benchmark_details\.reference_|other_references\..*\.reference_'),
    ('benchmark', r'\.benchmark$|benchmark_details\.benchmark$'),
    ('score', r'\.score$|benchmark_details\.score$'),
    ('provenance', r'\.provenance\.'),
]
def categorize(path):
    for name, rx in _CATS:
        if _re.search(rx, path):
            return name
    return 'other'
def group_differences(diffs):
    """diffs: [(zone, path, live, standalone, rel)] -> DataFrame grouped by zone, indicator, category."""
    import pandas as _pd
    rows = [{'zone': z, 'indicator': p.split('.')[0], 'category': categorize(p), 'rel': rel} for z, p, a, b, rel in diffs]
    df = _pd.DataFrame(rows)
    if df.empty: return df
    g = df.groupby(['zone', 'indicator', 'category'], sort=False).agg(fields=('rel', 'size'), max_relative_difference=('rel', 'max')).reset_index()
    return g

# ---- helpers for the reference-reproducibility diagnostic (definitions only; the evidence printed below is read from the standalone audits)
from darukaa_reference.reference import ReferenceSelector
_ORIG_TODAY = ReferenceSelector.__dict__['_today_str']            # the raw staticmethod object; restored after every pinned run
DATES = [(date(2026, 9, 27) + timedelta(days=i)).isoformat() for i in range(7)]       # 2026-09-27 .. 2026-10-03
_ZONES = {}
RESULT_KEYS = ['status', 'site_value', 'reference_n', 'reference_median', 'reference_mad', 'benchmark', 'score', 'reference_population']

def standalone_audit(label):
    return json.load(open(f'{STANDALONE_DIR}/{label}_audit.json'))
def standalone_row(label, indicator):
    return next(r for r in standalone_audit(label)['rows'] if r['indicator'] == indicator)
def eligible_of(r):
    return json.loads(r['provenance']).get('reference_construction', {}).get('eligible_cells_total')

def recorded_dates(label, indicator):
    """When the STANDALONE run built this reference, read from the audit itself: provenance.written_utc (when the audit was written) minus the engine time of every
    indicator evaluated after this one. Returns (candidate UTC dates, build start, build end). Two dates only if the build straddled midnight."""
    d = standalone_audit(label)
    written = datetime.fromisoformat(d['provenance']['written_utc'].replace('Z', '+00:00')).replace(tzinfo=None)
    rows = d['rows']; i = next(k for k, r in enumerate(rows) if r['indicator'] == indicator)
    end = written - timedelta(seconds=sum(r['seconds'] for r in rows[i + 1:])); start = end - timedelta(seconds=rows[i]['seconds'])
    return sorted({start.date().isoformat(), end.date().isoformat()}), start, end

def pinned_run(label, indicator, day):
    """Run ONE indicator of ONE zone through the frozen engine, with the date seen by the reference construction pinned to `day` (None = today). Restored afterwards."""
    spec = next(s for s in tata_specs if s.label == label)
    try:
        setattr(ReferenceSelector, '_today_str', _ORIG_TODAY if day is None else staticmethod(lambda d=day: d))
        used = ReferenceSelector._today_str()
        t0 = time.time()
        if label not in _ZONES:
            _ZONES[label] = A.load_zone(config, registry, spec.path, spec.label, spec.realm)
        rows = A.run_smoke_test(config, registry, _ZONES[label], out_dir=f'/content/diag_work/{label}_{indicator}_{day or "unpinned"}', only=[indicator], log=lambda *a, **k: None)
    finally:
        setattr(ReferenceSelector, '_today_str', _ORIG_TODAY)
    return used, next(x for x in rows if x['indicator'] == indicator), time.time() - t0

def compare_to_standalone(label, indicator, r):
    """EXACT = every row field identical (floats to rel 1e-12) except row 'seconds' and provenance.git_commit. The tolerance is the one used for parity; it is not changed."""
    sa = standalone_row(label, indicator)
    allnd = [d for k in sorted(set(r) | set(sa)) if k != 'seconds' for d in deep_diff(r.get(k), sa.get(k), f'{indicator}.{k}') if d[0] != f'{indicator}.provenance.git_commit']
    resnd = [d for k in RESULT_KEYS for d in deep_diff(r.get(k), sa.get(k), f'{indicator}.{k}')]
    if eligible_of(r) != eligible_of(sa): resnd.append((f'{indicator}.eligible_cells_total', eligible_of(r), eligible_of(sa), None))
    if not allnd and not resnd: return 'EXACT', allnd, resnd
    if not resnd: return f'RESULT FIELDS EXACT ({len(allnd)} other fields differ)', allnd, resnd
    return f'NO ({len(resnd)} result fields differ)', allnd, resnd

def table_row(label, indicator, tag, used, r, kind='pinned'):
    sa = standalone_row(label, indicator); m, allnd, resnd = compare_to_standalone(label, indicator, r)
    rec = recorded_dates(label, indicator)[0]
    return {'zone': label, 'indicator': indicator, 'date tested': tag, 'eligible reference cells (this run / standalone)': f'{eligible_of(r)} / {eligible_of(sa)}',
            'reference n': r['reference_n'], 'reference median': r['reference_median'], 'reference MAD': r['reference_mad'], 'benchmark': r['benchmark'], 'score': r['score'],
            'match to standalone audit': m, 'recorded in audit': 'YES' if (kind == 'pinned' and tag in rec) else '',
            '_eligible': eligible_of(r), '_exact': m == 'EXACT', '_recorded': kind == 'pinned' and tag in rec, '_kind': kind}

SHOW = ['zone', 'indicator', 'date tested', 'recorded in audit', 'eligible reference cells (this run / standalone)', 'reference n', 'reference median', 'reference MAD', 'benchmark', 'score',
        'match to standalone audit']
def show(rows):
    display(pd.DataFrame(rows)[SHOW])

print('\nThe standalone audits record when they were written (provenance.written_utc). Reference-build dates they imply (UTC, estimated from the per-indicator engine time):')
for _l in EXPECTED_TATA:
    for _i in ('ghm', 'chm', 'ndvi'):
        _r = standalone_row(_l, _i)
        if _r['status'] == 'scored':
            _d, _s, _e = recorded_dates(_l, _i)
            print(f'  {_l:20s} {_i:5s} built {_s:%Y-%m-%d %H:%M:%S} -> {_e:%H:%M:%S}  => date seen by the reference construction: {_d}')


## REFERENCE REPRODUCIBILITY DIAGNOSTIC (D1 -> D2 -> D3)

**What it tests.** The first live run matched the standalone audits on site values, statuses and `reference_n`, but the *reference populations* differed (fewer eligible cells; shifted median/MAD/benchmark/score) for exactly the indicators
whose reference is stratified by Dynamic World land cover (`ghm`, `chm`, `ndvi`). In the frozen engine that stratum is built from a window ending on **the date the code runs**, and the audits do not record that date as a result.
The audits do record when they were **written** (`provenance.written_utc`), which lets this notebook work out when each reference was built and test exactly that date.

**What it does.** It re-runs single indicators through the frozen engine changing **only the date seen by the reference construction** (restored afterwards). It does not modify any file, the engine, the tolerance or the methodology.
- **D1:** Pond_4 `ghm` at every date from 2026-09-27 to 2026-10-03 and once unpinned, plus a determinism control.
- **D2:** Pond_2, Pond_3 and EMU_Wetland_forest `ghm`, and EMU_Wetland_forest `chm`, each at its own recorded date and the day either side.
- **D3:** one master table and a conclusion: **A** historical reference reproduced by pinned date, **B** date dependence exists but does not reproduce it, **C** date dependence is not sufficient to explain it.

Run D1, then D2, then D3. Nothing to enter.

In [ ]:
# D1 - date-pinned GHM test on Pond_4. NOTHING TO ENTER. Runs ~5 minutes. For every date from 2026-09-27 to 2026-10-03 (and once with today's real date) it re-runs the frozen
# engine's ghm for Pond_4 with ONLY the date seen by the reference construction changed, and compares the result with the standalone audit.
DIAG_STARTED = time.time()
rec = recorded_dates('Pond_4', 'ghm')[0]
print('Pond_4 ghm: the standalone audit implies the reference was built on', rec, '(from provenance.written_utc)\n')
D1 = []
for day in DATES + [None]:
    used, r, secs = pinned_run('Pond_4', 'ghm', day)
    D1.append(table_row('Pond_4', 'ghm', day or 'unpinned (today: %s)' % used, used, r, kind='pinned' if day else 'unpinned'))
    print(f"  {day or 'unpinned':11s} -> date used {used}: eligible {D1[-1]['eligible reference cells (this run / standalone)']}  {D1[-1]['match to standalone audit']}  ({secs:.0f}s)")
# determinism control: repeat one date; if the same date does not give the same answer, the date cannot be the explanation
ctl_day = next((x['date tested'] for x in D1 if x['_exact'] and x['date tested'] in DATES), rec[0])
used, r, secs = pinned_run('Pond_4', 'ghm', ctl_day)
ctl = table_row('Pond_4', 'ghm', ctl_day + ' (repeat)', used, r, kind='control')
first = next(x for x in D1 if x['date tested'] == ctl_day)
CONTROL_SAME = all(ctl[k] == first[k] for k in ('_eligible', 'reference median', 'reference MAD', 'benchmark', 'score'))
D1.append(ctl)
print(f"\n  determinism control (same date twice): {'IDENTICAL' if CONTROL_SAME else 'DIFFERENT -> run-to-run nondeterminism; the date cannot be isolated'}")
show(D1)
print('\nStandalone Pond_4 ghm: eligible', eligible_of(standalone_row('Pond_4', 'ghm')), '| median', standalone_row('Pond_4', 'ghm')['reference_median'], '| benchmark', standalone_row('Pond_4', 'ghm')['benchmark'])
print('Dates that reproduce it EXACTLY:', [x['date tested'] for x in D1 if x['_exact'] and x['_kind'] == 'pinned'] or 'NONE')


In [ ]:
# D2 - cross-zone confirmation. NOTHING TO ENTER except (optionally) the one switch below. Each zone is tested at the date its OWN audit implies, and at the day before and the
# day after (controls: if the reference depends on the date, only the recorded date should reproduce the standalone value). ~10 minutes; ndvi (~40 min per date) is opt-in.
INCLUDE_NDVI = False        # the only switch: True also tests EMU_Wetland_forest ndvi at its recorded date (adds ~40 minutes)
assert 'D1' in globals(), 'run D1 first'
D2 = []
def around(label, ind):
    rec = recorded_dates(label, ind)[0]
    lo = (date.fromisoformat(rec[0]) - timedelta(days=1)).isoformat(); hi = (date.fromisoformat(rec[-1]) + timedelta(days=1)).isoformat()
    return sorted({lo, *rec, hi})
for label, ind in [('Pond_2', 'ghm'), ('Pond_3', 'ghm'), ('EMU_Wetland_forest', 'ghm'), ('EMU_Wetland_forest', 'chm')] + ([('EMU_Wetland_forest', 'ndvi')] if INCLUDE_NDVI else []):
    days = around(label, ind) if ind != 'ndvi' else recorded_dates(label, ind)[0]
    print(f'{label}/{ind}: recorded date(s) {recorded_dates(label, ind)[0]}; testing {days}')
    for day in days:
        used, r, secs = pinned_run(label, ind, day)
        D2.append(table_row(label, ind, day, used, r)); print(f"    {day}: eligible {D2[-1]['eligible reference cells (this run / standalone)']}  {D2[-1]['match to standalone audit']}  ({secs:.0f}s)")
show(D2)
if not INCLUDE_NDVI:
    print('\nndvi was NOT tested (INCLUDE_NDVI = False). It uses the same stratum code as chm; set the switch to True to confirm it.')


In [ ]:
# D3 - diagnostic summary and conclusion. NOTHING TO ENTER.
assert 'D1' in globals() and 'D2' in globals(), 'run D1 and D2 first'
ALL = D1 + D2
pinned = [x for x in ALL if x['_kind'] == 'pinned']
refs = {}
for x in pinned: refs.setdefault((x['zone'], x['indicator']), []).append(x)
dependence = {k: (len({x['_eligible'] for x in v}) > 1 or len({(x['reference median'], x['reference MAD'], x['benchmark']) for x in v}) > 1) for k, v in refs.items()}
reproduced = {k: any(x['_exact'] for x in v if x['_recorded']) for k, v in refs.items()}
exact_dates = {k: [x['date tested'] for x in v if x['_exact']] for k, v in refs.items()}
print('MASTER TABLE (every run of this diagnostic)\n'); show(ALL)
summary = pd.DataFrame([{'zone': k[0], 'indicator': k[1], 'reference changes with the date tested': dependence[k], 'reproduced EXACTLY at its recorded date': reproduced[k],
                         'dates that reproduce it exactly': exact_dates[k]} for k in refs])
print('\nPER REFERENCE\n'); display(summary)
if not CONTROL_SAME:
    outcome, text = 'C', 'date dependence is not sufficient to explain the discrepancy (the determinism control failed: the SAME date did not give the same result)'
elif all(reproduced.values()) and any(dependence.values()):
    outcome, text = 'A', 'historical reference reproduced by pinned date'
elif any(dependence.values()):
    outcome, text = 'B', 'date dependence exists but does not reproduce the historical reference (for every reference tested)'
else:
    outcome, text = 'C', 'date dependence is not sufficient to explain the discrepancy (results did not change with the date, yet differ from the standalone audit)'
print('\n' + '=' * 100); print(f'CONCLUSION: {outcome}. {text}'); print('=' * 100)
print('  references tested     :', sorted({f"{z}/{i}" for z, i in refs}))
print('  change with the date  :', sorted(f"{z}/{i}" for (z, i), v in dependence.items() if v))
print('  reproduced at recorded:', sorted(f"{z}/{i}" for (z, i), v in reproduced.items() if v))
if outcome == 'B': print('  NOT reproduced        :', sorted(f"{z}/{i}" for (z, i), v in reproduced.items() if not v))
partial = [x for x in pinned if x['_recorded'] and x['match to standalone audit'].startswith('RESULT FIELDS EXACT')]
if partial: print('  note: result fields are exact but other row fields differ for:', [(x['zone'], x['indicator'], x['date tested']) for x in partial])
print('  dates that reproduce each zone (they differ by zone):', {f"{z}/{i}": d for (z, i), d in exact_dates.items() if d})
print('\nWHAT THIS DOES NOT SHOW: bii, forest_loss_rate and cpland use the same stratum code and are predicted (not tested) to behave the same.' +
      ('' if any(i == 'ndvi' for _, i in refs) else ' ndvi was not tested.'))
print('No methodology, tolerance, engine file or configuration was changed by this diagnostic.')
SUMMARY_DIR = f'{OUT_ROOT}/diag_{RUN_TAG}'; os.makedirs(SUMMARY_DIR, exist_ok=True)
pd.DataFrame(ALL)[SHOW].to_csv(f'{SUMMARY_DIR}/reference_reproducibility_table.csv', index=False)
json.dump({'outcome': outcome, 'text': text, 'control_same': CONTROL_SAME, 'identity': IDENTITY, 'dependence': {f'{z}/{i}': v for (z, i), v in dependence.items()},
           'reproduced': {f'{z}/{i}': v for (z, i), v in reproduced.items()}, 'exact_dates': {f'{z}/{i}': v for (z, i), v in exact_dates.items()}},
          open(f'{SUMMARY_DIR}/reference_reproducibility_summary.json', 'w'), indent=1, default=str)
print('\nSaved automatically to', SUMMARY_DIR)


## Part A: the four live Tata zones through the pipeline (run LATER, after the diagnostic is resolved)

Takes about 95 minutes. Its parity gate compares every field with the standalone audits; at the moment it is expected to stop on the date-dependent references (see the diagnostic). Do not run Part B until Part A passes.

In [ ]:
# A1. Output folder for Part A (FRESH: a rerun needs a new RUN_TAG in cell 5, so an old result is never mistaken for this run)
OUT_T = f'{OUT_ROOT}/tata_4zone_{RUN_TAG}'
assert not os.path.exists(OUT_T) or not os.listdir(OUT_T), f'{OUT_T} is not fresh: change RUN_TAG in cell 5'
os.makedirs(OUT_T, exist_ok=True)
json.dump(IDENTITY, open(f'{OUT_T}/_run_identity.json', 'w'), indent=1)
print('output folder:', OUT_T, '| expected runtime: about 95 minutes (EMU_Wetland_forest is the long one)')

In [ ]:
# A2. RUN: manifest -> v0.2.9 pipeline -> frozen v0.2.8 engine (LIVE Earth Engine) -> adapter -> per-realm aggregation -> coverage/headline -> HTML.
#    continue_on_zone_failure=False: any zone failure stops the run loudly. strict_provenance=True (default).
RUN_START = time.time()
report_T = EP.run_engine_project(config, registry, 'TataMotors_Pimpri_live4', tata_specs, OUT_T, resume=False,
                                 continue_on_zone_failure=False, strict_provenance=True)
print(f'\nfinished in {(time.time() - RUN_START) / 60:.1f} min | zones: {list(report_T["zones"])} | failed: {report_T["meta"]["failed_zones"]}')

In [ ]:
# A3. PARITY (live audit files written by THIS run vs the standalone audits).
import pandas as pd
PARITY_PASSED = False
KEY_FIELDS = ['indicator', 'realm', 'status', 'applicability_reason', 'site_value', 'reference_n', 'reference_median', 'reference_mad', 'benchmark', 'score',
              'scoring_method', 'direction', 'reason']
summary, all_unexpected, tables = [], [], []
for label in EXPECTED_TATA:
    zrec = report_T['zones'][label]
    lf = zrec['audit_files']['json']
    assert not zrec['resumed_from_existing_audit'] and os.path.getmtime(lf) >= RUN_START, f'{label}: audit was not produced by this live run'
    live = json.load(open(lf)); sa = json.load(open(f'{STANDALONE_DIR}/{label}_audit.json'))
    assert live['git_commit'].startswith(commit[:7]) and sa['git_commit'].startswith(FROZEN_COMMIT_PREFIX)      # live = pipeline commit; standalone = frozen tag
    assert live['provenance']['engine']['engine_closure_sha256_at_import'] == FROZEN_ENGINE_CLOSURE_SHA256
    unexpected, expected, n = compare_audits(live, sa)
    all_unexpected += [(label,) + u for u in unexpected]
    summary.append({'zone': label, 'realm': live['rows'][0]['realm'], 'rows live/standalone': f"{len(live['rows'])}/{len(sa['rows'])}", 'fields compared': n,
                    'UNEXPECTED differences (B)': len(unexpected), 'expected differences (A)': len(expected)})
    sr = {r['indicator']: r for r in sa['rows']}
    for r in live['rows']:
        if r['status'] == 'scored' or sr[r['indicator']]['status'] == 'scored':
            s = sr[r['indicator']]
            tables.append({'zone': label, 'indicator': r['indicator'], 'status': r['status'], 'site live': r['site_value'], 'site standalone': s['site_value'],
                           'ref n': f"{r['reference_n']}/{s['reference_n']}", 'ref median live': r['reference_median'], 'ref median standalone': s['reference_median'],
                           'benchmark live': r['benchmark'], 'benchmark standalone': s['benchmark'], 'score live': r['score'], 'score standalone': s['score']})
    # the pipeline layer: the report's rows must carry the engine's live values unchanged (adapter pass-through)
    for r in zrec['rows']:
        lr_ = next(x for x in live['rows'] if x['indicator'] == r['indicator'])
        for k in KEY_FIELDS + ['site_unit', 'detail', 'flags', 'provenance', 'diagnostics']:
            assert deep_diff(r[k], lr_[k], k) == [], f"ADAPTER CHANGED an engine value: {label}/{r['indicator']}/{k}"
display(pd.DataFrame(summary))
print('\nEXPECTED (A) differences are only: row "seconds" and row provenance.git_commit. Everything else must be identical.\n')
display(pd.DataFrame(tables))
if all_unexpected:
    print('\n!!! UNEXPECTED ENGINE-LEVEL DIFFERENCES (B) -- STOP. Do not run Part B.\n')
    json.dump([{'zone': z, 'path': p, 'live': a, 'standalone': b, 'relative_difference': rel, 'category': categorize(p)} for z, p, a, b, rel in all_unexpected],
              open(f'{OUT_T}/_parity_differences.json', 'w'), indent=1, default=str)
    print(f'ALL {len(all_unexpected)} differences are saved in {OUT_T}/_parity_differences.json (none dropped). Grouped summary:\n')
    display(group_differences(all_unexpected))
    cat = pd.DataFrame([{'category': categorize(p)} for _, p, *_ in all_unexpected]).value_counts().rename('fields').reset_index()
    print('\nby category:'); display(cat)
    print('\nfirst 10 raw differences:')
    for z, path, a, b, rel in all_unexpected[:10]:
        print(f'  {z}: {path}  live={str(a)[:60]!r} standalone={str(b)[:60]!r}' + (f'  rel={rel:.3e}' if rel is not None else ''))
    raise RuntimeError(f'{len(all_unexpected)} unexpected engine-level difference(s): stopping before any broader validation')
PARITY_PASSED = True
print(f'ENGINE-LEVEL PARITY: all {sum(s["fields compared"] for s in summary)} row fields identical (tolerance rel {REL_TOL}) for {len(summary)} zones, apart from the 2 expected fields.')

In [ ]:
# A4 (definitions). Report / HTML checks (definitions only)
import re, html as htmllib
from darukaa_reference import html_report
def validate_report(report, out_dir, project_name):
    page = open(f'{out_dir}/{project_name}_project.html', encoding='utf-8').read()
    text = htmllib.unescape(re.sub(r'<[^>]+>', ' ', page))
    from darukaa_reference import engine_report as _ER
    plain = ' '.join(text.split())
    claims = plain.replace(' '.join(_ER.SCORE_EXPLAINER.split()), '')          # the disclaimer itself quotes "100 % condition": exclude it from the claim search
    csv_text = open(f'{out_dir}/{project_name}_project.csv', encoding='utf-8').read()
    res = []
    def chk(i, desc, ok, detail=''): res.append((i, desc, bool(ok), detail))
    rows = [r for z in report['zones'].values() for r in z['rows']]
    n_scored = sum(r['aggregation_role'] == 'aggregate' for r in rows)
    n_ok = sum(i['status'] == 'ok' for rl in report['realms'].values() for i in rl['aggregation']['per_indicator'].values())
    bold = re.findall(r'<b>(\d\.\d\d)</b>', page)
    chk(1, 'raw measurement, reference median and benchmark score are separate labelled columns',
        all(h in page for h in ('Benchmark score (0-1)', 'Raw measurement', 'Reference median')))
    import csv as _csv, io as _io
    cs = list(_csv.DictReader(_io.StringIO(csv_text)))
    chk(2, 'n/a is never a number: score cells exist ONLY for scored rows (HTML) and every unscored CSV row has an empty score/benchmark/site cell',
        len(bold) == n_scored + n_ok and all(r['score'] == '' and r['benchmark'] == '' for r in cs if r['status'] != 'scored')
        and all(r['site_value'] == '' for r in cs if r['status'] in ('not_applicable', 'pending_methodology')),
        f'bold score cells {len(bold)} vs scored rows {n_scored} + realm indicators {n_ok}')
    leak = [(rl, i['indicator'], z) for rl, rr in report['realms'].items() for i in rr['aggregation']['per_indicator'].values()
            for z in i.get('zone_scores', {}) if next(r for r in report['zones'][z]['rows'] if r['indicator'] == i['indicator'])['aggregation_role'] != 'aggregate']
    chk(3, 'contextual / screening / excluded rows never enter a condition aggregate', not leak, str(leak[:3]))
    miss = [l for l, z in report['zones'].items() if not re.search(re.escape(l) + r'\s*\(' + z['realm'] + ',', plain)]
    chk(4, 'realm is explicit: a section per realm and the realm in every zone heading',
        all(f'{r.title()} realm' in page for r in report['realms']) and not miss, f'zones without realm in heading: {miss}')
    chk(5, 'coverage is visible: pillar coverage table per realm', page.count('Pillar coverage by zone') == len(report['realms']))
    bad6 = []
    for rl, rr in report['realms'].items():
        h = rr['headline']; pz = h['pillar_zone_coverage']
        if not h['available']:
            ok = 'no overall condition is reported' in text
        else:
            ok = set(rr['profile']['components']) <= {p for p, v in pz.items() if v['zones_scored'] > 0}
            ok = ok and f"{h['condition']['score']:.2f}" in text
        if not ok: bad6.append(rl)
    chk(6, 'headline: withheld without condition evidence; a pillar with no scored zone never carries a score', not bad6, f'{bad6}')
    badges = re.findall(r'dk-badge-score">([^<]*)<', page)
    chk(7, 'no score shown as a percentage / "100% biodiversity"; score explainer present',
        ' '.join(_ER.SCORE_EXPLAINER.split()) in plain and not any('%' in b for b in badges)
        and not re.search(r'100\s*%\s*(biodiversity|ecological|condition|intactness)', claims, re.I) and not re.search(r'<b>\d\.\d\d\s*%', page),
        f'badges {badges}')
    chk(8, 'zone-level evidence accessible: a section per zone and the audit files on disk',
        page.count('<details class="dk-collapsible">') == len(report['zones']) and
        all(os.path.exists(p) for z in report['zones'].values() for p in z['audit_files'].values()))
    chk(9, 'existing Darukaa report visual family retained (same CSS and class names)',
        html_report._CSS in page and 'dk-header' in page and 'dk-table' in page and '--dk-green' in page)
    miss10 = []
    for rl, rr in report['realms'].items():
        cav = rr['headline']['coverage_caveat']
        if cav:
            a, b = page.find(f'{rl.title()} realm'), page.find('Pillar coverage by zone', page.find(f'{rl.title()} realm'))
            sec = page[a:b] if a >= 0 and b > a else ''
            if not sec or (htmllib.escape(cav) not in sec and htmllib.escape(cav, quote=False) not in sec): miss10.append(rl)
    chk(10, 'every coverage caveat is displayed WITH its headline (before the coverage table), not only in JSON', not miss10, f'{miss10}')
    chk(11, 'engine identity shown in the report footer and the engine is the frozen one', '(NOT FROZEN)' not in page and 'engine sha256' in page)
    return res

def show_checks(res):
    print(pd.DataFrame([{'#': i, 'check': d, 'result': 'PASS' if ok else 'FAIL', 'detail': det} for i, d, ok, det in res]).to_string(index=False))
    return all(ok for _, _, ok, _ in res)

In [ ]:
# A4. Report validation for Part A, then look at the report.
checks_T = validate_report(report_T, OUT_T, 'TataMotors_Pimpri_live4')
REPORT_CHECKS_PASSED = show_checks(checks_T)
assert REPORT_CHECKS_PASSED, 'a report check failed: STOP and send the table back'
for realm, rr in report_T['realms'].items():
    h = rr['headline']
    print(f"\n== {realm}: {rr['n_zones']} zones | headline available: {h['available']}")
    if h['available']:
        print('   condition benchmark score', h['condition']['score'], '| class', h['condition']['concern_class'], '| pressure', h['pressure']['score'], '| decision cell', h['matrix_cell'])
    else:
        print('  ', h['no_headline_reason'])
    print('   coverage:', h['coverage_caveat'])
from IPython.display import HTML, display
display(HTML(f'<iframe srcdoc="{htmllib.escape(open(OUT_T + "/TataMotors_Pimpri_live4_project.html", encoding="utf-8").read())}" width="100%" height="900"></iframe>'))
print('\nPART A COMPLETE. Read the parity table and the report above. Only if you accept them, continue to Part B.')

## Part B: SoulForest, seven zones, all declared terrestrial

This runs only after Part A parity and report checks have passed. There is **no standalone v0.2.8 baseline for SoulForest**, so this does **not** test parity; it tests that the
same pipeline and frozen engine, driven only by the manifest, produce a complete, correctly separated, correctly labelled result for a different project.
`EMU_ANCHOR_Wetland` is **terrestrial** (the water has receded). If the engine's own applicability rules activate a riparian indicator for it, that is expected and stays inside the terrestrial realm.

In [ ]:
# B1. PART B setup. Hard-stops unless Part A passed.
assert PARITY_PASSED and REPORT_CHECKS_PASSED, 'Part A has not passed: do not run Part B'
EXPECTED_SOUL = {l: 'terrestrial' for l in ['EMU_ANCHOR_Fruit_Forest', 'EMU_ANCHOR_Rock_Guild', 'EMU_ANCHOR_Wetland', 'SEG01', 'SEG02', 'SEG03', 'SEG04']}
_warn.clear(); logging.getLogger('darukaa_reference.manifest').addHandler(_h)
soul_project, soul_specs = M.resolve_zone_specs(M.find_manifest_by_project_name(REPO_ROOT, 'SoulForest_Veltoor'), REPO_ROOT)
logging.getLogger('darukaa_reference.manifest').removeHandler(_h)
assert not _warn, f'legacy inference was used: {_warn}'
assert [s.label for s in soul_specs] == list(EXPECTED_SOUL), [s.label for s in soul_specs]
for s in soul_specs:
    assert s.realm == 'terrestrial' and s.realm_source == 'manifest' and s.companion_source is None, (s.label, s.realm, s.realm_source)
    print(f'{s.label:26s} realm={s.realm} declared by: {s.realm_source}')
assert next(s for s in soul_specs if s.label == 'EMU_ANCHOR_Wetland').realm == 'terrestrial'
OUT_S = f'{OUT_ROOT}/soulforest_7zone_{RUN_TAG}'
assert not os.path.exists(OUT_S) or not os.listdir(OUT_S), f'{OUT_S} is not fresh: change RUN_TAG'
os.makedirs(OUT_S, exist_ok=True); json.dump(IDENTITY, open(f'{OUT_S}/_run_identity.json', 'w'), indent=1)
print('\n7 zones, 41 ha in total; output folder:', OUT_S)

In [ ]:
# B2. RUN SoulForest: same pipeline, same frozen engine, live Earth Engine.
RUN_START_S = time.time()
report_S = EP.run_engine_project(config, registry, 'SoulForest_Veltoor', soul_specs, OUT_S, resume=False, continue_on_zone_failure=False, strict_provenance=True)
print(f'\nfinished in {(time.time() - RUN_START_S) / 60:.1f} min | zones: {len(report_S["zones"])} | failed: {report_S["meta"]["failed_zones"]}')

In [ ]:
# B3. SoulForest checks (integrity and behaviour; NOT parity: no standalone baseline exists).
from collections import Counter
assert list(report_S['realms']) == ['terrestrial'], list(report_S['realms'])
assert all(z['realm'] == 'terrestrial' and {r['realm'] for r in z['rows']} == {'terrestrial'} and len(z['rows']) == 46 for z in report_S['zones'].values())
assert all(r['status'] == 'not_applicable' and r['reason'] == 'domain_mismatch' for z in report_S['zones'].values() for r in z['rows'] if r['indicator'] in ('sabf', 'wcpi', 'sdi', 'tspi'))
checks_S = validate_report(report_S, OUT_S, 'SoulForest_Veltoor')
SOUL_CHECKS_PASSED = show_checks(checks_S)
print('\nstatus counts per zone:')
display(pd.DataFrame({l: Counter(r['status'] for r in z['rows']) for l, z in report_S['zones'].items()}).fillna(0).astype(int))
wet = next(r for r in report_S['zones']['EMU_ANCHOR_Wetland']['rows'] if r['indicator'] == 'riparian_natural_veg_share')
print('\nEMU_ANCHOR_Wetland realm =', report_S['zones']['EMU_ANCHOR_Wetland']['realm'], '| riparian_natural_veg_share:', wet['status'], wet['reason'], '| score', wet['score'])
h = report_S['realms']['terrestrial']['headline']
print('\nheadline available:', h['available'], '| condition', h['condition']['score'], h['condition']['concern_class'], '| pressure', h['pressure']['score'])
print('coverage:', h['coverage_caveat'])
display(HTML(f'<iframe srcdoc="{htmllib.escape(open(OUT_S + "/SoulForest_Veltoor_project.html", encoding="utf-8").read())}" width="100%" height="900"></iframe>'))
assert SOUL_CHECKS_PASSED

In [ ]:
# B4. What was written. Everything is on Google Drive; send back the printed tables from cells 4, D3, A3, A4 and B3.
for d in (OUT_T, OUT_S):
    print(d); [print('   ', f) for f in sorted(os.listdir(d))]
print('\nNothing above is a validation result by itself: it is evidence for you to review.')